In [ ]:
# STEP 1: Import libraries and define file paths
# Description: Set up required libraries and define key directory/file paths

import os
import pandas as pd

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
models_base_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

# Fuel columns (must match exactly your specification)
fuel_types = [
    "electricity",
    "lpg",
    "mineral_wood",
    "mains_gas",
    "oil",
    "wood_logs",
    "wood_pellets",
    "wood_chips",
    "coal"
]

In [2]:
# STEP 2: Load EPC dataset
# Description: Read EPC data and ensure correct columns exist

epc_df = pd.read_csv(epc_file)

required_columns = ["BUILDING_REFERENCE_NUMBER", "MAIN_FUEL_CODE", "SECOND_FUEL_CODE"]

for col in required_columns:
    if col not in epc_df.columns:
        raise ValueError(f"Missing required column: {col}")

# Ensure fuel codes are lowercase for consistency
epc_df["MAIN_FUEL_CODE"] = epc_df["MAIN_FUEL_CODE"].str.lower()
epc_df["SECOND_FUEL_CODE"] = epc_df["SECOND_FUEL_CODE"].str.lower()

In [3]:
# STEP 3: Iterate through buildings and process heat_load_dis.csv
# Description: Loop through each building, check for file existence, and generate heat_by_fuel.csv

for _, row in epc_df.iterrows():
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    main_fuel = row["MAIN_FUEL_CODE"]
    second_fuel = row["SECOND_FUEL_CODE"]

    heating_folder = os.path.join(models_base_path, building_id, "HEATING")
    input_file = os.path.join(heating_folder, "heat_load_dis.csv")
    output_file = os.path.join(heating_folder, "heat_by_fuel.csv")

    # Skip if heat_load_dis.csv does not exist
    if not os.path.exists(input_file):
        print(f"Skipping {building_id}: heat_load_dis.csv not found")
        continue

    try:
        heat_df = pd.read_csv(input_file)
    except Exception as e:
        print(f"Error reading {building_id}: {e}")
        continue

    # Validate required columns in heat_load_dis.csv
    required_heat_cols = ["Time", "secondary_heat_power", "main_heat_power"]
    for col in required_heat_cols:
        if col not in heat_df.columns:
            print(f"Skipping {building_id}: missing column {col}")
            continue

    # Create output dataframe with Time column
    output_df = pd.DataFrame()
    output_df["Time"] = heat_df["Time"]

    # Initialize all fuel columns with zeros
    for fuel in fuel_types:
        output_df[fuel] = 0.0

    # Add main heat power to correct fuel column
    if main_fuel in fuel_types:
        output_df[main_fuel] += heat_df["main_heat_power"]
    else:
        print(f"Warning: Unknown MAIN_FUEL_CODE '{main_fuel}' for {building_id}")

    # Add secondary heat power to correct fuel column
    if second_fuel in fuel_types:
        output_df[second_fuel] += heat_df["secondary_heat_power"]
    else:
        print(f"Warning: Unknown SECOND_FUEL_CODE '{second_fuel}' for {building_id}")

    # Save output CSV
    try:
        output_df.to_csv(output_file, index=False)
        print(f"Processed {building_id}")
    except Exception as e:
        print(f"Error saving {building_id}: {e}")

Processed 1001829067
Skipping 1001951858: heat_load_dis.csv not found
Skipping 1001829071: heat_load_dis.csv not found
Skipping 1234761001: heat_load_dis.csv not found
Skipping 1001991633: heat_load_dis.csv not found
Processed 1001664929
Processed 1001829059
Skipping 1001063639: heat_load_dis.csv not found
Skipping 1234761000: heat_load_dis.csv not found
Skipping 1236594950: heat_load_dis.csv not found
Skipping 1001664925: heat_load_dis.csv not found
Skipping 1001906271: heat_load_dis.csv not found
Processed 1000238911
Skipping 1001829057: heat_load_dis.csv not found
Skipping 1234760999: heat_load_dis.csv not found
Skipping 1002143357: heat_load_dis.csv not found
Skipping 1001951854: heat_load_dis.csv not found
Skipping 1001829069: heat_load_dis.csv not found
Skipping 1002313096: heat_load_dis.csv not found
Skipping 1002143351: heat_load_dis.csv not found
Processed 1001870854
Skipping 1001870864: heat_load_dis.csv not found
Skipping 1002143293: heat_load_dis.csv not found
Skipping 1002

In [4]:
# STEP 4: Summary check (optional)
# Description: Count how many heat_by_fuel.csv files were successfully created

count = 0

for _, row in epc_df.iterrows():
    building_id = str(row["BUILDING_REFERENCE_NUMBER"])
    output_file = os.path.join(models_base_path, building_id, "HEATING", "heat_by_fuel.csv")
    
    if os.path.exists(output_file):
        count += 1

print(f"Total heat_by_fuel.csv files created: {count}")

Total heat_by_fuel.csv files created: 25
